# VDS in depth: composition, formats, and clusters

The [vds notebook](vds.ipynb) covers the basic create → read → append loop. This
one goes further:

1. virtualize **another plugin's output** (Harmony subset) with `assets=`
2. why pydap output cannot be virtualized
3. store-to-store merges with `source=`
4. the manifest form (`load=False`) for both store formats
5. kerchunk-parquet specifics (versioned hrefs, `replace=True`)
6. running the opens on a Dask cluster, and credential delivery
7. the parser / VCC authorization escape hatches

Every request has the same shape:

```python
ea.plugins.vds.request(results_or_none, **options) -> Job
```

In [ ]:
import earthaccess as ea

ea.login()

## Install

```bash
pip install "earthaccess-plugin-vds[distributed]"

# To compose with Harmony (see below). Install it without deps for now: its
# pyproject pins the contract at @main, while vds needs the vds-contract branch.
uv pip install --no-deps -e ../earthaccess-plugin-harmony
uv pip install "harmony-py>=0.6"
```

In [ ]:
import time


def wait_for(job, timeout=600):
    """Block until a job settles; print the reason if it failed."""
    status = job.wait(timeout=timeout)
    if status.name in {"FAILED", "CANCELED"}:
        print(job.info())
    return status

## 1. Compose with Harmony

Harmony writes a **new file** (the subset), so the natural workflow is:
subset → take the returned `Asset`s → virtualize them. Pass them with
`assets=`; no search results are needed.

Request a file format the parsers understand (`application/x-netcdf4`). A
Harmony job usually takes under a minute.

In [ ]:
results = ea.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-02", "2013-01-02"), count=1
)

harmony_job = ea.plugins.harmony.request(
    results,
    variables=["analysed_sst"],
    output_format="application/x-netcdf4",
    skip_preview=True,
)
wait_for(harmony_job)

subset_assets = harmony_job.results()
subset_assets

The result is a normal `Asset`:

```
Asset(href='https://harmony.earthdata.nasa.gov/service-results/..._subsetted.nc4',
      access='https', auth='edl', format='netcdf4')
```

vds sees `https` + `netcdf4`, picks `HDFParser` (derived files have no DMR++
sidecars), and uses your EDL token for the NASA host. With `destination_url`
pointing at `s3://…`, it would use your environment AWS credentials instead.

In [ ]:
subset_store = "harmony_subset.icechunk"

job = ea.plugins.vds.request(
    None,                       # no granules: the assets ARE the input
    assets=subset_assets,
    store=subset_store,
    concat_dim="time",
)
wait_for(job)
asset = job.results()[0]
asset

Appending works exactly like the granule path: another Harmony subset, then
the same request against the existing store.

In [ ]:
next_results = ea.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-03", "2013-01-03"), count=1
)
next_job = ea.plugins.harmony.request(
    next_results,
    variables=["analysed_sst"],
    output_format="application/x-netcdf4",
    skip_preview=True,
)
wait_for(next_job)

append_job = ea.plugins.vds.request(
    None, assets=next_job.results(), store=asset.href, concat_dim="time"
)
wait_for(append_job)

ds = ea.open_virtual(asset.href)
print(ds.sizes, ds.time.values)

## 2. Plain URLs, and why pydap cannot be composed

`assets=` also accepts URL strings, e.g. the original granule files (useful to
virtualize a hand-picked list, or files produced by another tool).

In [ ]:
granules = ea.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-04", "2013-01-05"), count=2
)
urls = [granule.data_links(access="indirect")[0] for granule in granules]

url_job = ea.plugins.vds.request(None, assets=urls, concat_dim="time")
wait_for(url_job)
url_vds = url_job.results()
print(url_vds.sizes, type(url_vds["analysed_sst"].data).__name__)

**pydap is different.** Its assets are OPeNDAP URLs with a constraint
expression — a server-side response, not a file with byte ranges. There is
nothing for a virtual store to point at, so vds rejects them. (Subset with
Harmony, or virtualize the original granules.)

In [ ]:
from earthaccess_plugin_spec import Asset

opendap_asset = Asset(href="https://example.com/dap4?var=x", access="opendap")
try:
    ea.plugins.vds.request(None, assets=[opendap_asset], concat_dim="time")
except ValueError as exc:
    print("rejected:", exc)

## 3. Store-to-store merge: `source=`

`source=` is for **virtual stores**, not files: open an existing store
(icechunk or kerchunk) and merge it into a new target. The existing store is
reopened in manifest form (`IcechunkParser` / `KerchunkParquetParser`) — no
data is copied. Works with either format for both source and target.

In [ ]:
merged_job = ea.plugins.vds.request(
    store="merged.icechunk",
    source=subset_store,       # the Harmony-built store from section 1
    concat_dim="time",
)
wait_for(merged_job)
merged = merged_job.results()[0]

print(ea.open_virtual(merged.href).sizes)

## 4. The manifest form: `load=False`

`load=True` (the default) returns the lazy, concrete xarray view. `load=False`
returns the **manifest form**: `ManifestArray` variables pointing at byte
ranges, with concrete coordinates — the same view the appender uses. It works
for both formats now.

In [ ]:
from virtualizarr.manifests import ManifestArray

references = ea.open_virtual(subset_store, load=False)
name = next(iter(references.data_vars))
print(name, "->", type(references[name].data).__name__)
assert isinstance(references[name].data, ManifestArray)
print(f"{references.virtualize.nbytes / 1e6:.1f} MB referenced, 0 copied")

You can combine manifests in memory and write the result yourself:

```python
new = ea.plugins.vds.request(new_granules, concat_dim="time").results()
combined = xr.combine_nested([references, new], concat_dim="time")
```

In practice prefer `source=` / `store=`, which do this without touching data.

## 5. Kerchunk-parquet specifics

A kerchunk store is a references **directory** named `*.parquet`. There is no
in-place append, so appending reads the old references, combines them with the
delta, and writes a **new versioned directory**. That means the href changes —
keep the latest `Asset`.

In [ ]:
kerchunk_job = ea.plugins.vds.request(
    granules, store="granules_refs.parquet", format="kerchunk", concat_dim="time"
)
wait_for(kerchunk_job)
kerchunk_asset = kerchunk_job.results()[0]
print(kerchunk_asset.href)

more = ea.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-06", "2013-01-06"), count=1
)
append = ea.plugins.vds.request(
    more, store=kerchunk_asset.href, format="kerchunk", concat_dim="time"
)
wait_for(append)
kerchunk_asset = append.results()[0]
print(kerchunk_asset.href)          # ..._refs_v2.parquet

print(ea.open_virtual(kerchunk_asset.href).sizes)

Pass `replace=True` to rewrite the original directory in place instead (stable
href, e.g. for a published URL). `commit=` is icechunk-only.

## 6. Running on a Dask cluster

Pass an existing `distributed.Client`; the per-granule opens are submitted to
it. This is generic Dask — a local cluster, Coiled, `dask-jobqueue`, …

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, processes=True)
client = Client(cluster)
client

Workers do the reads, so **credentials must exist where they run**:
`~/.netrc`, `EARTHDATA_TOKEN`, or `EARTHDATA_USERNAME`/`EARTHDATA_PASSWORD`.
The token is never part of the task payload: the registry ships as a
credential-free config and each worker builds its own stores. Pass the client
to any request:

In [ ]:
cluster_job = ea.plugins.vds.request(
    granules,
    concat_dim="time",
    client=client,
    show_progress=True,
)
wait_for(cluster_job)
print(cluster_job.info()["dashboard"])

For a **remote** cluster, deliver the environment explicitly. Coiled's
`send_private_envs` is the only private channel; a plain Dask cluster requires
`private=False` (values then go through the scheduler).

In [ ]:
# Remote / Coiled example — never put secrets in plugin payloads:
#
# ea.distribute_credentials(
#     {
#         "AWS_ACCESS_KEY_ID": ...,
#         "AWS_SECRET_ACCESS_KEY": ...,
#         "AWS_SESSION_TOKEN": ...,
#     },
#     cluster=cluster,            # Coiled: private channel
#     # private=False,            # plain Dask: explicit opt-in
# )
#
# The result reports the variables it set, never the values:
# {'transport': 'coiled', 'keys': [...], 'scheduler': ..., 'workers': 4}

## 7. Progress, cancellation, parsers, and VCCs

`job.info()` always has the same shape; `phase` moves
`virtualize → combine → write → done`, and with a client the counters come from
the cluster's futures.

In [ ]:
info = cluster_job.info()
print({key: info[key] for key in ("phase", "opened", "total", "failed", "percent")})

# Cancels per-granule opens that have not started yet (client.cancel with a client)
# cluster_job.cancel()

**Parsers.** The default is `DMRPPParser` (NASA DMR++ sidecars), with an
automatic `HDFParser` fallback. `assets=` defaults straight to `HDFParser`.
Explicit choices and combine controls:

In [ ]:
# parser="HDFParser", group="/", drop_variables=["mask"], combine="nested"
job = ea.plugins.vds.request(
    granules,
    concat_dim="time",
    parser="HDFParser",
    drop_variables=["mask"],
)
wait_for(job)
job.results().data_vars

**VCC authorization.** Reading a store authorizes its virtual chunk containers
with your EDL credentials. For a custom endpoint or bucket, override it
explicitly (icechunk credentials, never serialized by the plugin):

In [ ]:
# import icechunk
# ds = ea.open_virtual(
#     asset.href,
#     authorize_virtual_chunk_access={
#         "s3://my-bucket/": icechunk.s3_credentials(access_key_id=..., secret_access_key=...),
#     },
# )

## Reference

| Option | Meaning |
|---|---|
| `store` | icechunk path/URI or kerchunk `*.parquet` directory |
| `format` | `"icechunk"` (default) or `"kerchunk"` |
| `assets` | file `Asset`s / URLs to virtualize (Harmony output); not with `source` |
| `source` | virtual store to merge into `store` |
| `append_dim` | defaults to `concat_dim` when the store exists |
| `replace` | kerchunk only: rewrite in place |
| `commit` | icechunk only |
| `access` | `"indirect"` (HTTPS) or `"direct"` (in-region S3) |
| `parser`, `group`, `concat_dim`, `combine`, `preprocess`, `drop_variables`, `loadable_variables` | VirtualiZarr / xarray controls |
| `load` | with no store: concrete dataset; with a store: rejected (use `ea.open_virtual(..., load=True)`) |
| `client` | existing `distributed.Client` for the opens |
| `show_progress` | per-granule `tqdm` bar |

## Clean up

Close the client and cluster; stores on disk are untouched.

In [ ]:
client.close()
cluster.close()